# AudioLDM: kuş referansı ile audio-to-audio

Resmî `audioldm-m-full` checkpoint'inde pozitif koşul referans waveform → CLAP audio embedding'dir. Kullanıcı text prompt'u verilmez. Boş metin unconditional branch içinde kalır; model tamamen text bileşenlerinden arındırılmış değildir.

Tek aday üretilir; text ile candidate ranking kapalıdır. Eski conditioner'ın eval sırasında da uyguladığı %10 koşul dropout'u 0 yapılır. Referansın enerji eğrisi veya timbresini birebir koruma garantisi yoktur. Bu bir bağımsız RMS/pitch/brightness kontrol deneyi değildir.

Referans: önceki 04–05'te kullanılan InspectorJ Robin kaydının lisanslı librosa OGG türevi. Kaynak SHA-256 doğrulanır.


## Ayrı model ortamı

Notebook mevcut proje kernel'inde kalabilir; model ayrı `.venv-audioldm` Python sürecinde çalışır. Terminalde, repository root'ta, bir kez:

```powershell
python -m venv .venv-audioldm
.\.venv-audioldm\Scripts\python.exe -m pip install torch==2.11.0 torchaudio==2.11.0 torchvision==0.26.0 --index-url https://download.pytorch.org/whl/cu130
.\.venv-audioldm\Scripts\python.exe -m pip install -r requirements-audioldm.txt
```

Bu kurulum `.venv` içindeki T-FOLEY/CUDA paketlerini değiştirmez. CPU/Linux için aynı üçlü `cpu` index'inden kurulabilir. İlk model koşumu yaklaşık **4.57 GB** resmî checkpoint ve küçük tokenizer/config dosyalarını indirir. Kaynak, checkpoint ve tokenizer revizyon/checksum kontrolü yapılır. Windows/GPU koşumu henüz doğrulanmadı.


In [ ]:
from pathlib import Path
import sys, os, json, subprocess
import numpy as np
import librosa
import soundfile as sf
import matplotlib.pyplot as plt
from IPython.display import Audio, Markdown, display

ROOT = Path.cwd()
if not (ROOT / "src/baselines").exists():
    ROOT = ROOT.parent
assert (ROOT / "src/baselines/audioldm_reference_inference.py").exists()
sys.path.insert(0, str(ROOT))
from src.audio.real_sample import load_robin, ATTRIBUTION

AUDIOLDM_PYTHON = ROOT / ".venv-audioldm" / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
assert AUDIOLDM_PYTHON.exists(), "Önce ayrı ortam kurulumunu tamamlayın."
subprocess.run([str(AUDIOLDM_PYTHON), "-c", "import torch, torchaudio, torchvision; print(torch.__version__, torchaudio.__version__, torchvision.__version__, torch.cuda.is_available())"], check=True)
reference, reference_sr, reference_path = load_robin(ROOT)
ATTRIBUTION

from src.audio.audioldm_diagnostics import save_diagnostics


## Gerçek referansı dinleme

Bu player tek referansı normalize eder. Sonraki karşılaştırma iki sese ortak playback gain uygular; ölçülen/kaydedilen veriler değişmez.


In [ ]:
display(Markdown("**Gerçek referans: InspectorJ Robin kuş kaydı**"))
display(Audio(reference, rate=reference_sr, normalize=True))


## Üretim

50 DDIM adımı, seed 42, guidance 2.5; 5 saniyelik çıktı. Orijinal kayıt yaklaşık 2.70 saniyedir. Resmî referans önişlemesine uygun mono/16 kHz, mean centering, peak .5 ve crop/zero padding kullanılır. Koşullama waveform'u 5.12 saniyeye hazırlanır; vocoder'ın ek kuyruğu çıktıda 5 saniyeye kırpılır. Bu önişleme T-FOLEY'deki normalize edilmemiş RMS referansından farklıdır.


In [ ]:
STEM = "local_robin_reference"
command = [str(AUDIOLDM_PYTHON), str(ROOT / "src/baselines/audioldm_reference_inference.py"),
           "--prepare", "--reference", str(reference_path), "--device", "auto",
           "--steps", "50", "--duration", "5", "--seed", "42", "--output-name", STEM]
subprocess.run(command, cwd=ROOT, check=True)


## Referans ve çıktı

Aynı playback gain, bağımsız player normalizasyonu kapalı. Orijinal referans ve çıktı süreleri farklıdır. Benzer ses içeriği, birebir waveform veya onset kopyası anlamına gelmez.


İlk CPU çıktısı düşük seviyeli bulundu. Ortak gain player'larından sonra, yalnız içerik dinlemesi için ayrıca normalize edilmiş çıktı player'ı verilir; bu üçüncü player seviye karşılaştırması için kullanılmaz.

In [ ]:
folder = ROOT / "results/audio/audioldm"
reference_16k, sr = sf.read(folder / f"{STEM}_reference.wav")
generated, generated_sr = sf.read(folder / f"{STEM}.wav")
assert sr == generated_sr == 16000
common_peak = max(float(np.max(np.abs(reference_16k))), float(np.max(np.abs(generated))), np.finfo(float).eps)
for label, samples in [("Gerçek kuş referansı", reference_16k), ("AudioLDM — referans ses koşullu çıktı", generated)]:
    display(Markdown(f"**{label}**"))
    display(Audio(samples / common_peak, rate=sr, normalize=False))
display(Markdown("**Çıktı — ayrı normalize edilmiş içerik dinlemesi**"))
display(Audio(generated, rate=sr, normalize=True))
report = json.loads((ROOT / f"experiments/baselines/audioldm/{STEM}_run.json").read_text(encoding="utf-8"))
report


## Zaman–frekans incelemesi

Aynı FFT/hop ve sabit FFT-magnitude dB renk aralığı (pencere kazancına göre kalibre edilmiş dBFS değil); gerçek süreler korunur. Spektrogram benzerliği otomatik kategori veya algısal kalite skoru değildir. Bu modelin doğrudan onset/RMS eğrisi kontrolü sunmadığı için T-FOLEY RMS hata skorlarıyla sıralama yapılmaz.


In [ ]:
summary, fig = save_diagnostics(ROOT, STEM)
display(summary)
plt.show()
